In [6]:
import pandas as pd
from typing import Dict, List, Tuple, Optional

def load_dataset(
    path: str,
    sep: str = ";",
    encoding: str = "utf8",
) -> pd.DataFrame:
    # si tu CSV es problemático, cambia engine="python" y on_bad_lines="warn"
    return pd.read_csv(path, sep=sep, encoding=encoding)

def add_cite_column(
    df: pd.DataFrame,
    id_col: str = "Article_ID",
    cite_col: str = "cite",
    prefix: str = r"\cite{",
    suffix: str = "}",
    position: str = "after_id"  # "first", "last", "after_id"
) -> pd.DataFrame:
    """
    Crea/actualiza la columna `cite` con el formato: \\cite{Article_ID}.
    Opcionalmente la inserta en una posición específica.
    """
    if id_col not in df.columns:
        raise KeyError(f"No existe la columna '{id_col}' en el DataFrame.")

    cite_series = prefix + df[id_col].astype(str).str.strip() + suffix

    if cite_col in df.columns:
        df[cite_col] = cite_series
        return df

    # insertar columna en una posición útil
    if position == "first":
        df.insert(0, cite_col, cite_series)
    elif position == "last":
        df[cite_col] = cite_series
    elif position == "after_id":
        if id_col in df.columns:
            idx = df.columns.get_loc(id_col) + 1
            df.insert(idx, cite_col, cite_series)
        else:
            df[cite_col] = cite_series
    else:
        df[cite_col] = cite_series

    return df

def split_by_groups(
    df: pd.DataFrame,
    groups: Dict[str, Tuple[str, ...]],
    core_cols: Optional[List[str]] = None,
    keep_core_in_each_split: bool = True
) -> Dict[str, pd.DataFrame]:

    if core_cols is None:
        core_cols = ["Article_ID", "Title", "Year", "Venue", "DOI_or_URL"]

    core_cols = [c for c in core_cols if c in df.columns]

    assigned_cols = set(core_cols) if keep_core_in_each_split else set()
    splits: Dict[str, pd.DataFrame] = {}

    def cols_matching_prefixes(prefixes: Tuple[str, ...]) -> List[str]:
        return [c for c in df.columns if any(c.startswith(p) for p in prefixes)]

    for group_name, prefixes in groups.items():
        g_cols = cols_matching_prefixes(prefixes)
        g_cols = [c for c in g_cols if c not in assigned_cols]  # por si hay solapes

        cols = (core_cols + g_cols) if keep_core_in_each_split else g_cols
        splits[group_name] = df[cols].copy()

        assigned_cols.update(g_cols)

    rest_cols = [c for c in df.columns if c not in assigned_cols]
    splits["RESTO"] = df[rest_cols].copy()
    return splits

def split_d1_d2_d3_eval(
    df: pd.DataFrame,
    core_cols: Optional[List[str]] = None,
    keep_core_in_each_split: bool = True
) -> Dict[str, pd.DataFrame]:
    groups = {
        "D1": ("D1_",),
        "D2": ("D2_",),
        "D3": ("D3_",),
        "EVAL": ("Eval_", "QA_"),
    }
    return split_by_groups(df, groups, core_cols=core_cols, keep_core_in_each_split=keep_core_in_each_split)

def summarize_splits(splits: Dict[str, pd.DataFrame]) -> None:
    for name, dfx in splits.items():
        print(f"{name}: filas={len(dfx):,} cols={len(dfx.columns):,}")

# 1) cargar df
df = load_dataset("assets/sintesis/dataset.csv", sep=",", encoding="utf8")

# 2) crear columna cite en el DF principal (para que viaje a todos los splits)
df = add_cite_column(df, id_col="Article_ID", cite_col="cite", position="after_id")

# 3) split (incluye cite porque está en core_cols)
splits = split_d1_d2_d3_eval(
    df,
    core_cols=["Article_ID", "cite", "Title", "Year", "Venue", "DOI_or_URL"],
    keep_core_in_each_split=True
)

summarize_splits(splits)

# 4) variables finales
df_d1, df_d2, df_d3 = splits["D1"], splits["D2"], splits["D3"]
df_eval, df_resto = splits["EVAL"], splits["RESTO"]


D1: filas=19 cols=15
D2: filas=19 cols=18
D3: filas=19 cols=13
EVAL: filas=19 cols=15
RESTO: filas=19 cols=9


In [7]:
df_d1.columns

Index(['Article_ID', 'cite', 'Title', 'Year', 'Venue', 'DOI_or_URL',
       'D1_Pollutants_inputs', 'D1_Env_variables_inputs', 'D1_Target_output',
       'D1_Target_output_airpollution_class', 'D1_Reported_input_values',
       'D1_Sampling_frequency_values', 'D1_Time_window_values',
       'D1_Data_source_type', 'D1_Preprocessing'],
      dtype='str')

In [8]:
import pandas as pd
import re
from typing import Iterable, List, Optional

_NR_SET = {"NR", "N/R", "NA", "N/A", ""}

_SPLIT_RE = re.compile(r"[;,\|/]+")

def _clean_cell(x) -> str:
    if pd.isna(x):
        return ""
    s = str(x).strip()
    if not s or s.upper() in _NR_SET:
        return ""
    return s

def _normalize_list_like(s: str) -> str:
    """
    Si viene algo como 'PM2.5; PM10, NO2' lo normaliza a 'PM2.5 | PM10 | NO2'.
    Si no parece lista, lo deja igual.
    """
    if not s:
        return ""
    parts = [p.strip() for p in _SPLIT_RE.split(s) if p.strip()]
    if len(parts) <= 1:
        return s
    # dedup manteniendo orden
    seen = set()
    out = []
    for p in parts:
        key = p.lower()
        if key not in seen:
            seen.add(key)
            out.append(p)
    return " | ".join(out)

def build_synthesis_string(
    df: pd.DataFrame,
    cols: Optional[List[str]] = None,
    row_sep: str = "\n\n---\n\n",
    field_sep: str = "\n",
    kv_sep: str = ": ",
    include_header: bool = True,
    row_id_col: str = "Article_ID"
) -> str:
    """
    Devuelve un string único concatenando TODAS las filas.
    Para cada fila: (opcional) encabezado con Article_ID y luego 'Col: valor' por cada col relevante.
    """
    if cols is None:
        cols = [c for c in df.columns if c.startswith("D1_")]

    # Mantener solo columnas existentes
    cols = [c for c in cols if c in df.columns]

    blocks: List[str] = []
    for _, r in df.iterrows():
        fields: List[str] = []

        if include_header and row_id_col in df.columns:
            rid = _clean_cell(r[row_id_col])
            if rid:
                fields.append(f"[{rid}]")

        for c in cols:
            v = _clean_cell(r[c])
            if not v:
                continue
            v = _normalize_list_like(v)
            fields.append(f"{c}{kv_sep}{v}")

        # si la fila no aporta nada, sáltala
        if len(fields) > (1 if include_header and row_id_col in df.columns else 0):
            blocks.append(field_sep.join(fields))

    return row_sep.join(blocks)

# Ejemplo de uso con tu split D1:
# (elige exactamente qué columnas quieres incluir)




In [14]:
cols_d1 = [
    "cite",
    "D1_Pollutants_inputs",
    "D1_Env_variables_inputs",
    "D1_Target_output", 
    "D1_Reported_input_values"
    "D1_Data_source_type",
    "D1_Preprocessing",
]

d1_sinthesys = build_synthesis_string(
    df_d1,
    cols=cols_d1,
    include_header=True,
    row_id_col="Article_ID"
)

print(type(d1_sinthesys), len(d1_sinthesys))
print(d1_sinthesys)


<class 'str'> 4265
[A001]
cite: \cite{A001}
D1_Pollutants_inputs: CO2 | SO2 | NO2
D1_Env_variables_inputs: Temperature | Humidity | Wind
D1_Target_output: AQI reconstruction and pollutant concentration assessment
D1_Preprocessing: Calibration | Normalization

---

[A009]
cite: \cite{A009}
D1_Pollutants_inputs: CO | NOx | NH3 | Smoke
D1_Target_output: AQI prediction | class assignment
D1_Preprocessing: Other:Weekly_Aggregation

---

[A028]
cite: \cite{A028}
D1_Pollutants_inputs: NO2 | O3 | PM10 | PM2.5 | CO | SO2
D1_Env_variables_inputs: Temperature | Humidity | Air_Pressure | UV
D1_Target_output: AQI prediction
D1_Preprocessing: Cleaning | Feature_Engineering | Normalization

---

[A084]
cite: \cite{A084}
D1_Pollutants_inputs: PM2.5
D1_Env_variables_inputs: Temperature | Humidity | Air_Pressure
D1_Target_output: PM2.5 concentration adjustment | calibration against reference station
D1_Preprocessing: Synchronization | Missing_Imputation | Cleaning | Normalization

---

[A087]
cite: \cit

In [13]:
df_d1[['D1_Pollutants_inputs', 'D1_Env_variables_inputs',  'D1_Target_output_airpollution_class', 'D1_Target_output']]


,D1_Pollutants_inputs,D1_Env_variables_inputs,D1_Target_output_airpollution_class,D1_Target_output
0,CO2; SO2; NO2,Temperature; Humidity; Wind,AQI_IAQ_Estimation,AQI reconstruction and pollutant concentration...
1,CO; NOx; NH3; Smoke,NR,AQI_IAQ_Classification,AQI prediction/class assignment
2,NO2; O3; PM10; PM2.5; CO; SO2,Temperature; Humidity; Air_Pressure; UV,AQI_IAQ_Estimation,AQI prediction
3,PM2.5,Temperature; Humidity; Air_Pressure,Pollutant_Calibration_Adjustment,PM2.5 concentration adjustment/calibration aga...
4,TVOC,Temperature; Humidity,Pollutant_Prediction,VOC concentration prediction; Filter replaceme...
5,Smoke; Dust_particles,Temperature; Humidity,EXCLUDE_Control_Oriented,Environment quality classification; AC control...
6,CO; SO2,NR,EXCLUDE_Control_Oriented,Gas quality classification; Blower/Fan activation
7,PM,Temperature; Humidity,EXCLUDE_Control_Oriented,Air-Quality Power level / Heater output power
8,PM2.5; PM10; CO2; CO; NO2; TVOC,Temperature; Humidity,Pollutant_Prediction,PM2.5 forecasting
9,CO2; CO; PM2.5; PM10,Temperature; Humidity,AQI_IAQ_Classification,State of Indoor Air (SIA)


In [27]:
df_d2.columns

Index(['Article_ID', 'Title', 'Year', 'Venue', 'DOI_or_URL',
       'D2_Fuzzy_approach_type', 'D2_Fuzzy_role', 'D2_Model_size_values',
       'D2_Optimization_values', 'D2_Inference_purpose',
       'D2_Rule_base_definition', 'D2_Membership_functions',
       'D2_Defuzzification_method', 'D2_Risk_modeling_approach',
       'D2_Interpretability_elements'],
      dtype='str')

In [17]:
cols_single = [
    "D2_Fuzzy_approach_type",
    "D2_Fuzzy_role",
    "D2_Rule_base_definition",
    "D2_Defuzzification_method",
    "D2_Risk_modeling_approach",
    "D2_AQI_class_labels",
    "D2_AQI_class_count"
]

for col in cols_single:
    print(f"\n{col}")
    print(sorted(df_d2[col].dropna().unique()))



D2_Fuzzy_approach_type
['ADFIST', 'ADFIST; Mamdani', 'ANFIS', 'ANFIS; Mamdani; Other:TSK', 'ANFIS; Other:RANFIS', 'ANFIS; Sugeno', 'Hybrid_Fuzzy', 'I-ANFIS', 'Mamdani', 'Mamdani; Hybrid_Fuzzy', 'Mamdani; Sugeno', 'Other:Fuzzy_ARTMAP', 'Sugeno', 'Type1_FIS', 'Type1_FIS; Other:Tsukamoto']

D2_Fuzzy_role
['Central', 'Complementary']

D2_Rule_base_definition
['Adaptive_Rule_Base', 'Equation_Based', 'Explicit_Rule_Table', 'Other:Mamdani_Rule_Base', 'Other:Two_Rules_Reported']

D2_Defuzzification_method
['Centroid', 'NR', 'Other:Takagi_Sugeno_and_Centroid_reported', 'Weighted_Average']

D2_Risk_modeling_approach
['AQI_Thresholds', 'LoP_m-AQI', 'NR', 'Other:OR_Risk_Group_0_to_13', 'Other:PM2.5_Forecast_Only', 'Other:Threshold_Based_Environment_Quality', 'Prediction_Based_Risk']

D2_AQI_class_labels
['Good; Medium; Bad; Very_Bad; Dangerous', 'Good; Moderate; Poor; Unhealthy; Hazardous', 'Good; Moderate; Poor; Very_Poor; Hazardous', 'Good; Moderate; Sensitive; Unhealthy; Harmful; Hazardous', '

In [19]:
cols_multi = [
    "D2_Inference_purpose",
    "D2_Membership_functions",
    "D2_Interpretability_elements"
]

def get_unique_multilabel(df, column, sep=";"):
    return sorted(
        set(
            label.strip()
            for cell in df[column].dropna()
            for label in str(cell).split(sep)
            if label.strip() != ""
        )
    )

for col in cols_multi:
    print(f"\n{col}")
    print(get_unique_multilabel(df_d2, col))



D2_Inference_purpose
['AQI_Classification', 'AQI_Estimation', 'Other:Filter_Replacement_Support', 'Other:HVAC_Power_Optimization', 'Other:Sensor_Calibration_Adjustment', 'Pollutant_Prediction', 'Risk_Alerting', 'Route_Optimization']

D2_Membership_functions
['Adaptive_MF', 'Gaussian', 'Implicit_MF', 'Linguistic_Labels', 'Other:Bell_Shaped', 'Other:Generalized_Bell', 'Other:Linear_Output', 'Other:Trapezoidal', 'Other:Triangular', 'Other:Triangular_Trapezoidal']

D2_Interpretability_elements
['AQI_Mapping_Table', 'MF_Plots', 'NR', 'Other:Color_Mapping', 'Other:Confusion_Matrix', 'Other:Fuzzy_Surface', 'Other:Tree_Structure', 'Rule_Transparency', 'SOM_Visualization']


In [20]:
df_d3.columns

Index(['Article_ID', 'cite', 'Title', 'Year', 'Venue', 'DOI_or_URL',
       'D3_Platform_architecture', 'D3_Processing_mode',
       'D3_Communication_and_tech', 'D3_Visualization_alerting',
       'D3_Deployment_context', 'D3_Hardware_and_sensors',
       'D3_Real_time_characteristics'],
      dtype='str')